In [4]:
import kagglehub
import os

base_dir = 'D:\\UTE\\NAM3\\k2_dot_2\\xu_ly_ngon_ngu_tu_nhien\\cuoi_ky\\data'
path = kagglehub.dataset_download("haitranquangofficial/vietnamese-online-news-dataset")
input_path = os.path.join(path, 'news_dataset.json')
output_path = os.path.join(base_dir, 'news_dataset_processed.pkl')

d:\anaconda3\envs\it\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os

import urllib.request
url = "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/vietnamese-stopwords.txt"

# Nối đường dẫn gốc với tên file để lưu đúng chỗ
stopwords_path = os.path.join(base_dir, "vietnamese-stopwords.txt")

urllib.request.urlretrieve(url, stopwords_path)
print("2. Đã tải Stopwords! File được lưu tại:", stopwords_path)

2. Đã tải Stopwords! File được lưu tại: D:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky\data\vietnamese-stopwords.txt


In [6]:
import os
import urllib.request
import py_vncorenlp

model_dir = os.path.join(base_dir.replace('\\', '/'), 'models', 'wordsegmenter')
os.makedirs(model_dir, exist_ok=True)

files_to_download = {
    "VnCoreNLP-1.2.jar": "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar",
    "models/wordsegmenter/vi-vocab": "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab",
    "models/wordsegmenter/wordsegmenter.rdr": "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/wordsegmenter.rdr"
}

print("Đang kiểm tra các file mô hình VnCoreNLP...")
for file_path, url in files_to_download.items():
    local_path = os.path.join(base_dir, os.path.normpath(file_path))
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(url, local_path)
        print(f"Đã tải mới: {local_path}")
    else:
        print(f"Đã có sẵn: {local_path}")

print("\nĐang khởi tạo rdrsegmenter...")
if 'rdrsegmenter' not in globals():
    rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=base_dir)
    print("Hoàn tất khởi tạo mô hình mới!")
else:
    print("VnCoreNLP đã sẵn sàng từ trước, tiếp tục sử dụng mà không cần nạp lại!")



Đang kiểm tra các file mô hình VnCoreNLP...
Đã có sẵn: D:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky\data\VnCoreNLP-1.2.jar
Đã có sẵn: D:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky\data\models\wordsegmenter\vi-vocab
Đã có sẵn: D:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky\data\models\wordsegmenter\wordsegmenter.rdr

Đang khởi tạo rdrsegmenter...
Hoàn tất khởi tạo mô hình mới!


In [7]:
sample_text = "Sinh viên Công nghệ Thông tin đang làm đồ án cuối kỳ."
print("\nKết quả test tách từ:")

print(rdrsegmenter.word_segment(sample_text))


Kết quả test tách từ:
['Sinh_viên Công_nghệ_Thông_tin đang làm đồ_án cuối kỳ .']


In [8]:
import re
def normalize_date(text):

    pattern_dmy = r"\b(\d{1,2})[-/.](\d{1,2})[-/.](\d{2,4})\b"
    pattern_dm = r"\b(\d{1,2})[-/.](\d{1,2})\b"

    def repl_dmy(m):
        day, month, year = m.groups()

        if len(year) == 2:
            year = "20" + year

        return f" DATE{year}{int(month):02d}{int(day):02d} "

    def repl_dm(m):
        day, month = m.groups()

        return f" DATE{int(month):02d}{int(day):02d} "

    # xử lý ngày-tháng-năm trước
    text = re.sub(pattern_dmy, repl_dmy, text)

    # rồi mới xử lý ngày-tháng
    text = re.sub(pattern_dm, repl_dm, text)

    return text

In [9]:
import unicodedata
import string

def clean_text_keep_case(text):

    text = str(text)

    # Unicode normalize
    text = unicodedata.normalize("NFC", text)

    # HTML
    text = re.sub(r'<[^>]+>', ' ', text)

    # URL
    text = re.sub(r'http\S+', ' ', text)

    # Email
    text = re.sub(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
        ' ',
        text
    )

    # Date
    text = normalize_date(text)

    # Xóa dấu câu nhưng giữ "_" và "-"
    custom_punctuation = (
        string.punctuation
        .replace('_', '')
        .replace('-', '')
    )

    translator = str.maketrans(
        custom_punctuation,
        ' ' * len(custom_punctuation)
    )

    text = text.translate(translator)

    # Gom khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [10]:
def preprocess_pipeline_v2(text):

    if not isinstance(text, str):
        return ""

    if len(text.strip()) == 0:
        return ""

    cleaned = clean_text_keep_case(text)

    segmented_sentences = rdrsegmenter.word_segment(cleaned)

    # word_segment trả về list các câu dạng string
    tokenized_text = " ".join(segmented_sentences)

    tokenized_text = tokenized_text.lower()

    return tokenized_text

In [11]:
import pandas as pd

df = pd.read_json(input_path) 
df_prep2 = df.copy()

In [12]:
from tqdm.auto import tqdm
tqdm.pandas()

In [13]:
import time
start_time = time.time()

# Xử lý thiếu dữ liệu
df_prep2['title'] = df_prep2['title'].fillna('')
content_col = 'content'
df_prep2[content_col] = df_prep2[content_col].fillna('')

# Lọc bỏ các bài báo quá ngắn (rác)
df_prep2 = df_prep2[df_prep2[content_col].str.len() > 100].copy()

print(f" -> Số bài báo hợp lệ cần xử lý: {df_prep2.shape[0]}")

# Tiền xử lý Title và Content
print(" -> Đang xử lý cột 'title'...")
df_prep2['title_processed'] = df_prep2['title'].progress_apply(preprocess_pipeline_v2)

print(" -> Đang xử lý cột 'content'...")
df_prep2['content_processed'] = df_prep2[content_col].progress_apply(preprocess_pipeline_v2)

df_prep2['combined_processed'] = df_prep2['title_processed'] + " " + df_prep2['content_processed']

end_time = time.time()
print(f"\nXử lý xong trong {round(end_time - start_time, 2)} giây.")


print(f"\nĐang lưu dữ liệu đã xử lý vào: {output_path}...")
df_prep2.to_parquet(output_path, index=False) 
print("Hoàn tất toàn bộ quy trình!")

 -> Số bài báo hợp lệ cần xử lý: 160254
 -> Đang xử lý cột 'title'...


100%|██████████| 160254/160254 [00:37<00:00, 4322.70it/s]


 -> Đang xử lý cột 'content'...


100%|██████████| 160254/160254 [19:26<00:00, 137.39it/s]



Xử lý xong trong 1204.35 giây.

Đang lưu dữ liệu đã xử lý vào: D:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky\data\news_dataset_processed.pkl...
Hoàn tất toàn bộ quy trình!


### Tạo cột không dấu

In [14]:
import unicodedata

def remove_vn_accents(text):
    if not isinstance(text, str):
        return ""

    # Chuẩn hóa unicode về dạng tổ hợp (NFD) để tách riêng chữ cái và dấu
    text = unicodedata.normalize('NFD', text)

    # Lọc bỏ các ký tự dấu (Mn viết tắt Mark, Nonspacing)
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])

    # 'đ' trong tiếng Việt là một ký tự độc lập, không phải 'd' thêm dấu nên phải replace thủ công
    text = text.replace('đ', 'd').replace('Đ', 'D')

    return text

In [15]:
df_prep2['combined_unaccented'] = df_prep2['combined_processed'].apply(remove_vn_accents)

In [16]:
df_prep2[df_prep2['id'] == 218229]['combined_unaccented'].iloc[0]

'sao viet date0801 sao viet date0801 nsnd tu long hao_huc cham con trai moi sinh sao viet ngay date0801 nsnd tu long chia_se khoanh_khac cham_soc cau con trai vua chao_doi ca nha ta cung yeu_thuong nhau xa la nha gan nhau la cuoi anh hai_huoc chu_thich cho buc anh thuy_ngoc'

In [17]:
import os

output_pickle = os.path.join(base_dir, 'news_dataset_df_prep2.pkl')

df_prep2.to_pickle(output_pickle)

In [18]:
import pandas as pd

local_path = 'news_dataset_df_prep2.pkl'

try:
    df_load = pd.read_pickle(local_path)
    print(f"Đã tải thành công DataFrame")
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file")
except Exception as e:
    print(f"Lỗi không xác định: {e}")

Đã tải thành công DataFrame


In [19]:
df_load.head()

,id,author,content,picture_count,processed,source,title,topic,url,crawled_at,title_processed,content_processed,combined_processed,combined_unaccented
0,218270,,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",3,0,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,2022-08-01 09:09:22.817308,tên cướp tiệm vàng tại huế là đại_uý công_an c...,chiều date0731 công_an tỉnh thừa_thiên - huế đ...,tên cướp tiệm vàng tại huế là đại_uý công_an c...,ten cuop tiem vang tai hue la dai_uy cong_an c...
1,218269,(Nguồn: Sina),"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",1,0,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,2022-08-01 09:09:21.181469,bỏ_qua mạng 5g nga tiến thẳng từ 4g lên 6g,gần đây thứ_trưởng bộ phát_triển kỹ_thuật_số t...,bỏ_qua mạng 5g nga tiến thẳng từ 4g lên 6g gần...,bo_qua mang 5g nga tien thang tu 4g len 6g gan...
2,218268,Hồ Sỹ Anh,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,3,0,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,2022-08-01 09:09:15.311901,địa_phương nào đứng đầu cả nước tổng điểm 3 mô...,kết_quả thi tốt_nghiệp thpt năm 2022 cho thấy ...,địa_phương nào đứng đầu cả nước tổng điểm 3 mô...,dia_phuong nao dung dau ca nuoc tong diem 3 mo...
3,218267,Ngọc Ánh,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,1,0,vnexpress,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,2022-08-01 09:09:02.211498,người chết trong mưa_lũ nghìn năm có một ở mỹ ...,thống_đốc kentucky_andy_beshear hôm date0731 c...,người chết trong mưa_lũ nghìn năm có một ở mỹ ...,nguoi chet trong mua_lu nghin nam co mot o my ...
4,218266,HẢI YẾN - MINH LÝ,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,12,0,soha,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,2022-08-01 09:09:01.601170,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...,vụ tai_nạn giao_thông liên_hoàn trên phố đi bộ...,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...,hai_phong hinh_anh xe dien gay tai_nan lien_ho...


In [20]:
df_prep3 = df_load.copy()

try:
    with open('vietnamese-stopwords.txt', 'r', encoding='utf-8') as f:
        stopwords_vn = set([line.strip().replace(' ', '_') for line in f])
    print(f"Đã tải thành công {len(stopwords_vn)} stopwords")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file vietnamese-stopwords.txt")
    stopwords_vn = set() 

def remove_stopwords(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    tokens = text.split()
    filtered_tokens = [token for token in tokens if token.lower() not in stopwords_vn]

    return " ".join(filtered_tokens)

print("Đang tiến hành loại bỏ stopwords...")

df_prep3['title_processed'] = df_load['title_processed'].astype(str).apply(remove_stopwords)
df_prep3['content_processed'] = df_load['content_processed'].astype(str).apply(remove_stopwords)

df_prep3['combined_processed'] = df_prep3['title_processed'] + " " + df_prep3['content_processed']


Đã tải thành công 1942 stopwords
Đang tiến hành loại bỏ stopwords...


In [21]:
output_pkl_path = 'news_dataset_df_prep3.pkl'
df_prep3.to_pickle(output_pkl_path)

### Chunking

In [22]:
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm

tqdm.pandas()

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def chunk_document_raw(title, content, window_size=4, overlap=1):
    title = str(title).strip()
    content = str(content).strip()
    sentences = sent_tokenize(content)

    if len(sentences) <= window_size:
        return [title + ". " + content]

    chunks = []
    i = 0
    while i < len(sentences):
        chunk_sentences = sentences[i : i + window_size]
        chunk_text = title + ". " + " ".join(chunk_sentences)
        chunks.append(chunk_text)
        i += (window_size - overlap)
        if i >= len(sentences) - overlap:
            break
    return chunks

In [23]:
df_prep = df.copy()

df_prep['title'] = df_prep['title'].fillna('')
content_col = 'content'
df_prep[content_col] = df_prep[content_col].fillna('')

original_len = len(df_prep)
df_prep = df_prep[df_prep[content_col].str.len() > 100].copy()

df_prep = df_prep.reset_index(drop=True)
df_prep['doc_id'] = df_prep.index.astype(str)

print(f"Đã lọc bỏ {original_len - len(df_prep)} bài rác. Giữ lại {len(df_prep)} bài báo hợp lệ.")

export_chunks = []    # Dành cho Chunk-level (GĐ4 & GĐ3 Hybrid)

tqdm.pandas(desc="Đang xử lý Document")

# Duyệt qua từng bài báo
for index, row in tqdm(df_prep.iterrows(), total=len(df_prep), desc="Processing Pipeline"):
    doc_id = row['doc_id']
    url = row.get('url', '')     # Lấy URL nếu có
    topic = row.get('topic', '') # Lấy Topic nếu có
    title = row['title']
    content = row[content_col]

    raw_chunks = chunk_document_raw(title, content, window_size=4, overlap=1)

    for chunk_idx, raw_chunk_text in enumerate(raw_chunks):
        chunk_id = f"{doc_id}_{chunk_idx}"

        chunk_processed = preprocess_pipeline_v2(raw_chunk_text)
        chunk_unaccented = remove_vn_accents(chunk_processed)

        export_chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "url": url,
            "topic": topic,
            "raw_text": raw_chunk_text,
            "chunk_processed": chunk_processed,
            "chunk_unaccented": chunk_unaccented
        })


Đã lọc bỏ 24285 bài rác. Giữ lại 160254 bài báo hợp lệ.


Processing Pipeline: 100%|██████████| 160254/160254 [32:35<00:00, 81.94it/s] 


In [27]:
import json


json_path = 'corpus_chunks.json'

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(export_chunks, f, ensure_ascii=False, indent=2)

print(f"Đã lưu thành công {len(export_chunks)} Chunks vào '{json_path}'")

Đã lưu thành công 1078604 Chunks vào 'corpus_chunks.json'


In [29]:
import pandas as pd

df_chunks = pd.DataFrame(export_chunks)

pkl_path = 'news_dataset_chunked.pkl'
df_chunks.to_pickle(pkl_path)

print(f"Đã chuyển thành DataFrame và lưu vào '{pkl_path}' thành công!")

Đã chuyển thành DataFrame và lưu vào 'news_dataset_chunked.pkl' thành công!


Inverted

In [30]:
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm

df_pred4 = df_load.copy()
def build_optimized_index(df_pred4):
    inverted_index = defaultdict(list)
    doc_lengths = {}

    ids = df_pred4['id'].values
    contents = df_pred4['combined_processed'].values

    print("Đang xây dựng chỉ mục... (có thể mất vài phút)")
    for i in tqdm(range(len(df_pred4))):
        doc_id = ids[i]
        tokens = str(contents[i]).split()
        doc_lengths[doc_id] = len(tokens)
        unique_tokens = set(tokens)
        for token in unique_tokens:
            inverted_index[token].append(doc_id)

    return inverted_index, doc_lengths

inverted_index, doc_lengths = build_optimized_index(df_pred4)

print(f"\nĐã xây dựng xong index với {len(inverted_index)} từ khóa duy nhất.")

Đang xây dựng chỉ mục... (có thể mất vài phút)


100%|██████████| 160254/160254 [00:28<00:00, 5677.12it/s]


Đã xây dựng xong index với 519282 từ khóa duy nhất.


In [34]:
import json
import numpy as np

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {int(k) if isinstance(k, (np.integer, int)) else str(k): convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    return obj

with open("inverted_index.json", "w", encoding="utf-8") as f:
    json.dump(convert_numpy(dict(inverted_index)), f, ensure_ascii=False, indent=2)

with open("doc_lengths.json", "w", encoding="utf-8") as f:
    json.dump(convert_numpy(doc_lengths), f, ensure_ascii=False, indent=2)

print("Đã lưu JSON.")

Đã lưu JSON.


In [35]:
import pickle

with open("inverted_index.pkl", "wb") as f:
    pickle.dump(dict(inverted_index), f)

with open("doc_lengths.pkl", "wb") as f:
    pickle.dump(doc_lengths, f)

print("Đã lưu PKL.")

Đã lưu PKL.
